<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/23%EC%9E%A5_%EB%94%A5%EB%9F%AC%EB%8B%9D%20%EA%B8%B0%EB%B0%98%20%EA%B0%9C%EC%B2%B4%EB%AA%85%20%EC%9D%B8%EC%8B%9D/%5B23-1%5D%20Bi-directional%20LSTM%EC%9D%84%20%EC%9D%B4%EC%9A%A9%ED%95%9C%20%EA%B0%9C%EC%B2%B4%EB%AA%85%20%EC%9D%B8%EC%8B%9D_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [23-1] Bi-directional LSTM을 이용한 개체명 인식 (수정본)

자연어처리 바이블 23장 실습 [23-1]을 최신 Colab(TensorFlow 2.16+ / Keras 3)에서 실행되도록 고친 노트북입니다. CoNLL-2003 영어 데이터로 **Embedding → Bidirectional LSTM → TimeDistributed(Dense)** 개체명 인식 모델을 학습합니다(교재 23.2 단어 단위 구조).

**원본 대비 수정 사항**
1. **Keras 3 대응 — `Tokenizer` 직접 구현**: `keras.preprocessing.text.Tokenizer`가 Keras 3에서 삭제되어, 같은 규칙(소문자화 · 빈도순 번호 · 상위 4,999개만 사용 · 나머지는 `OOV`=1)으로 단어/태그 사전을 직접 만듭니다. 결과 번호는 원본과 같습니다(예: 첫 문장 `[989, 1, 205, 629, 7, 3939, 216, 1, 3]`).
2. **import 경로 수정**: `pad_sequences`, `to_categorical`을 `keras.utils`에서 가져옵니다.
3. **Keras 3 인자 변경 반영**: `Embedding(input_length=...)` 삭제, `Adam(0.001)` → `Adam(learning_rate=0.001)`, 학습 기록 키 `'acc'` → `'accuracy'`.
4. **데이터 자동 다운로드**: `train.txt`를 업로드하지 않아도 GitHub에서 바로 받습니다.
5. **평가 보강**: 원본의 정확도는 대부분이 `O` 태그라 성능이 높게 보입니다. **패딩을 뺀 토큰 정확도**와 개체 단위 **정밀도·재현율·F1(seqeval)**을 추가했습니다.
6. **예측 확인 셀 추가**: 새 문장의 개체명을 예측해 봅니다. `random_state`·시드를 고정했습니다.

※ 4절까지는 원본과 같이 1 epoch만 학습합니다. 1 epoch 모델은 정확도는 0.83이지만 거의 모든 토큰을 `O`로 예측해 개체 단위 F1이 매우 낮습니다. 7절에서 4 epoch를 더 학습해 차이를 확인합니다.

## 1. 환경 설정

In [1]:
!pip install -q seqeval

import re
import urllib.request
from collections import Counter

import numpy as np
import keras
from keras import layers
from keras.utils import pad_sequences, to_categorical   # 수정: keras.preprocessing → keras.utils
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(555)
print("keras version:", keras.__version__)

keras version: 3.15.1


## 2. 데이터 불러오기

CoNLL-2003 형식: 한 줄에 `단어 품사 청크 개체명`, 빈 줄은 문장 구분, `-DOCSTART-`는 문서 구분입니다. 단어는 소문자로 바꾸고 (단어, 개체명 태그) 쌍만 사용합니다.

In [2]:
# 수정: 데이터 파일을 GitHub에서 바로 내려받기(업로드 불필요)
URL = ('https://raw.githubusercontent.com/mrdantzig/nlp-bible-code/master/'
       '23%EC%9E%A5_%EB%94%A5%EB%9F%AC%EB%8B%9D%20%EA%B8%B0%EB%B0%98%20'
       '%EA%B0%9C%EC%B2%B4%EB%AA%85%20%EC%9D%B8%EC%8B%9D/train.txt')
urllib.request.urlretrieve(URL, 'train.txt')

sentences = []
buff_sentence = []
with open('train.txt', 'r', encoding='utf-8') as f_r:
    for sentence in f_r.readlines():
        if len(sentence) == 0 or sentence.startswith('-DOCSTART-') or sentence[0] == "\n":
            if len(buff_sentence) > 0:
                sentences.append(buff_sentence)
                buff_sentence = []
            continue
        splits = sentence.split(' ')
        splits[-1] = re.sub(r'\n', '', splits[-1])   # 줄바꿈 제거
        word = splits[0].lower()                     # 단어는 소문자로
        buff_sentence.append([word, splits[-1]])     # (단어, 개체명 태그)
if buff_sentence:                                    # 수정: 파일 끝 문장도 포함
    sentences.append(buff_sentence)

sentences_info, tags_info = [], []
for sent in sentences:
    word, tag = zip(*sent)
    sentences_info.append(list(word))
    tags_info.append(list(tag))

print("문장 수:", len(sentences_info))
print("첫 번째 문장:", sentences_info[0])
print("첫 번째 태그 정보:", tags_info[0])

문장 수: 14041
첫 번째 문장: ['eu', 'rejects', 'german', 'call', 'to', 'boycott', 'british', 'lamb', '.']
첫 번째 태그 정보: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


## 3. 정수 인코딩과 패딩

원본의 `Tokenizer(num_words=5000, oov_token='OOV')`와 똑같이 동작하는 사전을 만듭니다.
- 번호 0은 패딩, 1은 `OOV`, 2부터 빈도가 높은 순서
- `num_words=5000`이므로 번호가 5000 미만인 단어만 쓰고, 나머지는 `OOV`(1)로 바꿉니다.

In [3]:
class SimpleTokenizer:
    """keras.preprocessing.text.Tokenizer(리스트 입력)와 같은 규칙의 간단한 구현"""
    def __init__(self, num_words=None, oov_token=None):
        self.num_words, self.oov_token = num_words, oov_token

    def fit_on_texts(self, texts):
        counts = Counter()
        for seq in texts:
            counts.update(w.lower() for w in seq)
        # 빈도 내림차순(동률은 처음 나온 순서) — 원본 Tokenizer와 같은 정렬
        vocab = [w for w, _ in sorted(counts.items(), key=lambda x: -x[1])]
        if self.oov_token is not None:
            vocab = [self.oov_token] + vocab
        self.word_index = {w: i + 1 for i, w in enumerate(vocab)}
        self.index_word = {i: w for w, i in self.word_index.items()}

    def texts_to_sequences(self, texts):
        oov = self.word_index.get(self.oov_token)
        out = []
        for seq in texts:
            ids = []
            for w in seq:
                i = self.word_index.get(w.lower())
                if i is None or (self.num_words and i >= self.num_words):
                    if oov is not None:
                        ids.append(oov)
                else:
                    ids.append(i)
            out.append(ids)
        return out

vocab_size = 5000
src_tokenizer = SimpleTokenizer(num_words=vocab_size, oov_token='OOV')   # 문장 데이터
src_tokenizer.fit_on_texts(sentences_info)
tar_tokenizer = SimpleTokenizer()                                         # 태그 데이터
tar_tokenizer.fit_on_texts(tags_info)

x_train = src_tokenizer.texts_to_sequences(sentences_info)
y_train = tar_tokenizer.texts_to_sequences(tags_info)
print(x_train[0])
print(y_train[0])

index2word = src_tokenizer.index_word
index2tag = tar_tokenizer.index_word
print("OOV 처리된 첫 문장:", [index2word[i] for i in x_train[0]])
print("태그 사전:", tar_tokenizer.word_index)

[989, 1, 205, 629, 7, 3939, 216, 1, 3]
[4, 1, 7, 1, 1, 1, 7, 1, 1]
OOV 처리된 첫 문장: ['eu', 'OOV', 'german', 'call', 'to', 'boycott', 'british', 'OOV', '.']
태그 사전: {'o': 1, 'b-loc': 2, 'b-per': 3, 'b-org': 4, 'i-per': 5, 'i-org': 6, 'b-misc': 7, 'i-loc': 8, 'i-misc': 9}


In [4]:
max_len = 80
x_train_padded = pad_sequences(x_train, padding='post', maxlen=max_len)
y_train_padded = pad_sequences(y_train, padding='post', maxlen=max_len)

x_train, x_test, y_train, y_test = train_test_split(
    x_train_padded, y_train_padded, test_size=.2, random_state=555)

tag_size = len(tar_tokenizer.word_index) + 1     # 9개 태그 + 패딩(0) = 10
y_train_oh = to_categorical(y_train, num_classes=tag_size)
y_test_oh = to_categorical(y_test, num_classes=tag_size)

print('훈련 샘플 문장의 크기: {}'.format(x_train.shape))
print('훈련 샘플 레이블의 크기: {}'.format(y_train_oh.shape))
print('테스트 샘플 문장의 크기: {}'.format(x_test.shape))
print('테스트 샘플 레이블의 크기: {}'.format(y_test_oh.shape))

훈련 샘플 문장의 크기: (11232, 80)
훈련 샘플 레이블의 크기: (11232, 80, 10)
테스트 샘플 문장의 크기: (2809, 80)
테스트 샘플 레이블의 크기: (2809, 80, 10)


## 4. Bi-LSTM 모델 학습

In [5]:
model = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True),  # 수정: input_length 삭제
    layers.Bidirectional(layers.LSTM(256, return_sequences=True)),
    layers.TimeDistributed(layers.Dense(tag_size, activation='softmax')),
])
model.compile(loss='categorical_crossentropy',
              optimizer=keras.optimizers.Adam(learning_rate=0.001),     # 수정: learning_rate=
              metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 80, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 80, 512)        │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 80, 10)         │         5,130 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,433,610 (5.47 MB)

 Trainable params: 1,433,610 (5.47 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
EPOCHS = 1   # 원본과 동일. 3~5로 늘리면 성능이 크게 오릅니다.
hist = model.fit(x_train, y_train_oh, batch_size=128, epochs=EPOCHS,
                 validation_data=(x_test, y_test_oh), verbose=2)

print("train loss: ", hist.history['loss'])
print("train accuracy: ", hist.history['accuracy'])     # 수정: 'acc' → 'accuracy'
print("val loss: ", hist.history['val_loss'])
print("val accuracy: ", hist.history['val_accuracy'])
print("\n 테스트 정확도: %.4f" % (model.evaluate(x_test, y_test_oh, verbose=0)[1]))

88/88 - 73s - 832ms/step - accuracy: 0.8260 - loss: 0.8277 - val_accuracy: 0.8329 - val_loss: 0.5575


train loss:  [0.8276945352554321]
train accuracy:  [0.8260494470596313]
val loss:  [0.5574944019317627]
val accuracy:  [0.832915723323822]



 테스트 정확도: 0.8329


## 5. 평가 보강 — 토큰 정확도와 개체 단위 F1

문장 대부분의 태그가 `O`라서, 모든 토큰을 `O`로만 예측해도 정확도가 높게 나옵니다. 개체명 인식은 보통 **개체 단위 F1**로 평가합니다(예: `New York` 전체를 `LOC`로 맞혀야 정답).

In [7]:
from seqeval.metrics import classification_report, f1_score

def evaluate_ner(model):
    pred = model.predict(x_test, batch_size=256, verbose=0).argmax(-1)
    true_tags, pred_tags = [], []
    for t_seq, p_seq in zip(y_test, pred):
        n = int((t_seq > 0).sum())                       # 패딩 제외 길이
        true_tags.append([index2tag[i].upper() for i in t_seq[:n]])
        pred_tags.append([index2tag.get(i, 'o').upper() if i > 0 else 'O' for i in p_seq[:n]])
    flat_t = sum(true_tags, []); flat_p = sum(pred_tags, [])
    token_acc = np.mean([a == b for a, b in zip(flat_t, flat_p)])
    all_o = np.mean([a == 'O' for a in flat_t])
    print(f"패딩 제외 토큰 정확도: {token_acc:.4f}  (모두 'O'로 예측할 때: {all_o:.4f})")
    print(f"개체 단위 F1: {f1_score(true_tags, pred_tags):.4f}\n")
    print(classification_report(true_tags, pred_tags, zero_division=0))

evaluate_ner(model)

패딩 제외 토큰 정확도: 0.8331  (모두 'O'로 예측할 때: 0.8294)
개체 단위 F1: 0.0364



              precision    recall  f1-score   support

         LOC       0.54      0.06      0.11      1379
        MISC       0.00      0.00      0.00       643
         ORG       0.01      0.00      0.00      1299
         PER       0.02      0.00      0.01      1319

   micro avg       0.18      0.02      0.04      4640
   macro avg       0.14      0.02      0.03      4640
weighted avg       0.17      0.02      0.04      4640



## 6. 새 문장 예측

In [8]:
def predict_ner(sentence):
    words = sentence.lower().split()
    ids = pad_sequences(src_tokenizer.texts_to_sequences([words]), padding='post', maxlen=max_len)
    p = model.predict(ids, verbose=0)[0].argmax(-1)
    for w, i in zip(sentence.split(), p[:len(words)]):
        print(f"{w:15s} {index2tag.get(int(i), 'o').upper()}")

predict_ner("EU rejects German call to boycott British lamb .")
print()
predict_ner("Peter Blackburn visited Germany and met officials from the European Commission .")

EU              B-PER
rejects         O
German          O
call            O
to              O
boycott         O
British         O
lamb            O
.               O

Peter           B-PER
Blackburn       O
visited         O
Germany         B-ORG
and             O
met             O
officials       O
from            O
the             O
European        O
Commission      O
.               O


## 7. 더 학습해 보기 (추가)

같은 모델을 4 epoch 더 학습(총 5 epoch)하고 다시 평가합니다. 정확도는 조금 오르지만 개체 단위 F1은 크게 오르는 것을 확인할 수 있습니다. (Colab GPU 기준 1분 내외)

In [9]:
model.fit(x_train, y_train_oh, batch_size=128, epochs=4,
          validation_data=(x_test, y_test_oh), verbose=2)
evaluate_ner(model)
predict_ner("EU rejects German call to boycott British lamb .")
print()
predict_ner("Peter Blackburn visited Germany and met officials from the European Commission .")

Epoch 1/4


88/88 - 66s - 746ms/step - accuracy: 0.8742 - loss: 0.4069 - val_accuracy: 0.9069 - val_loss: 0.3169


Epoch 2/4


88/88 - 79s - 902ms/step - accuracy: 0.9214 - loss: 0.2652 - val_accuracy: 0.9390 - val_loss: 0.2226


Epoch 3/4


88/88 - 85s - 960ms/step - accuracy: 0.9474 - loss: 0.1842 - val_accuracy: 0.9518 - val_loss: 0.1767


Epoch 4/4


88/88 - 65s - 740ms/step - accuracy: 0.9588 - loss: 0.1423 - val_accuracy: 0.9566 - val_loss: 0.1616


패딩 제외 토큰 정확도: 0.9566  (모두 'O'로 예측할 때: 0.8294)
개체 단위 F1: 0.7902



              precision    recall  f1-score   support

         LOC       0.88      0.86      0.87      1379
        MISC       0.79      0.67      0.73       643
         ORG       0.73      0.70      0.71      1299
         PER       0.87      0.76      0.81      1319

   micro avg       0.82      0.76      0.79      4640
   macro avg       0.82      0.75      0.78      4640
weighted avg       0.82      0.76      0.79      4640

EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O

Peter           B-PER
Blackburn       I-PER
visited         O
Germany         B-LOC
and             O
met             O
officials       O
from            O
the             O
European        B-ORG
Commission      I-ORG
.               O
